In [1]:
from pprint import pprint

import os
import re
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from joblib import dump
from rich.progress import track



## Data Processing

In [14]:
datadir = "C:/Users/acwh025/Documents/Software Dev/PTST-Visualiser/summaries"

test_summaries = [os.path.join(datadir, _) for _ in os.listdir(datadir)]

df = pd.DataFrame(columns=[
    'datalen_bytes', 
    'pub_count',
    'sub_count',
    'best_effort',
    'multicast',
    'durability',
    'mean_latency_us',
    'mean_total_throughput_mbps',
    'mean_sample_rate',
    'samples_received',
    'samples_lost'
])

for i in track(range( len(test_summaries) ), description="Processing summaries..."):
    summary = test_summaries[i]
    i = test_summaries.index(summary)
    summary_df = pd.read_csv(summary)
    testname = os.path.basename(summary.replace("_summary.csv", ""))
    datalen_bytes, pub_count, sub_count, best_effort, multicast, durability = get_settings_from_testname(testname)

    data = [
        datalen_bytes,
        pub_count,
        sub_count,
        best_effort,
        multicast,
        durability,
        summary_df["latency_us"].mean(),
        summary_df['total_throughput_mbps'].mean(),
        summary_df['total_sample_rate'].mean(),
        summary_df['total_samples_received'].max(),
        summary_df['total_samples_lost'].max()
    ]

    df.loc[i] = data
    
# Cut the df so all columns have the same length
min_length = len(df[df.columns[0]])
for col in df.columns:
    col_length = len(df[col])
    if col_length < min_length:
        min_length = col_length

# Cut the dataframe to the shortest column
df = df.iloc[:min_length]

df.to_csv('df.csv', index=False)

Output()

In [44]:
df = pd.read_csv('df.csv')

X = df[['datalen_bytes', 'pub_count', 'sub_count', 'best_effort', 'multicast', 'durability']]
y = df[['mean_latency_us', 'mean_total_throughput_mbps', 'mean_sample_rate', 'samples_received', 'samples_lost']]

# Splitting the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Creating the Random Forests model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)

# Training the model
rf_model.fit(X_train, y_train)

# Predicting on the test set
y_pred = rf_model.predict(X_test)

# Evaluating the model
model_r2_score = r2_score(y_test, y_pred)
print("R2 Score:", model_r2_score)

dump(rf_model, 'rf_model.joblib')

R2 Score: 0.904452844997915


['rf_model.joblib']